In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import pandas as pd
import numpy as np
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [2]:
# Set a fixed random seed to make model training and validation reproducible.
seed = 564563451

In [3]:
# Load the training and external validation datasets from CSV files.
train = pd.read_csv('train.csv')
X_ini = train.iloc[:,1:6]
y_ini = train.iloc[:,6]

In [4]:
# Create repeated stratified cross-validation splits for robust validation.
sp = RepeatedStratifiedKFold(random_state=seed,n_repeats=3,n_splits=5)

In [5]:
# Import the libraries required for data handling, modeling, and evaluation.
import hyperopt
from hyperopt import hp

In [6]:
# Define the hyperparameter objective function optimized by cross-validation AUC.
def objective(param):
    aucs = []
    for train_index,test_index in sp.split(X_ini,y_ini):
        X_train = X_ini.iloc[train_index,:]
        X_vali = X_ini.iloc[test_index,:]
        y_train = y_ini[train_index]
        y_vali = y_ini[test_index]
        model = SVC(random_state=seed,
                    C=param['C'],
                    gamma=param['gamma'],
                    probability=True)
        # Fit the model on the training cohort.
        model.fit(X_train,y_train)
        # Generate predicted probabilities for ROC-AUC evaluation.
        pro_vali = model.predict_proba(X_vali)[:,1]
        # Report model performance metrics for this cohort.
        auc_vali = roc_auc_score(y_vali,pro_vali)
        aucs.append(auc_vali)
    return -np.mean(aucs)

In [ ]:
# Specify the hyperparameter search space for Bayesian optimization.
space = {
    'C':hp.uniform('C',0,1),
    'gamma':hp.uniform('gamma',0,1),
}

In [8]:
# Run Bayesian hyperparameter optimization and keep the best parameter indices.
best_param = hyperopt.fmin(objective,space,hyperopt.tpe.suggest,max_evals=100)

100%|█████████████████████████████████████████████| 100/100 [00:18<00:00,  5.38trial/s, best loss: -0.8139756944444444]


In [9]:
# Display the best hyperparameter setting selected during optimization.
best_param

{'C': 0.571116670519715, 'gamma': 0.005816006215090652}

In [ ]:
model = SVC(random_state=seed,
            # Display the best hyperparameter setting selected during optimization.
            C=best_param['C'],
            gamma=best_param['gamma'],
            probability=True)
# Fit the model on the training cohort.
model.fit(X_ini,y_ini)
# Generate predicted probabilities for ROC-AUC evaluation.
pro_train = model.predict_proba(X_ini)[:,1]

In [ ]:
# Report model performance metrics for this cohort.
print('train set AUC={:.3f}'.format(roc_auc_score(y_ini,pro_train)))

In [ ]:
# Generate class labels for accuracy and related classification metrics.
y_pred_train = model.predict(X_ini)
# Report model performance metrics for this cohort.
auc_train = roc_auc_score(y_ini, pro_train)
accuracy_train = accuracy_score(y_ini, y_pred_train)
precision_train = precision_score(y_ini, y_pred_train)
recall_train = recall_score(y_ini, y_pred_train)
f1_train = f1_score(y_ini, y_pred_train)
confusion_train = confusion_matrix(y_ini, y_pred_train)
tn, fp, fn, tp = confusion_train.ravel()
specificity_train = tn / (tn + fp)

print('train set AUC={:.3f}'.format(auc_train))
print('train set Accuracy={:.3f}'.format(accuracy_train))
print('train set Precision={:.3f}'.format(precision_train))
print('train set Sensitivity (Recall)={:.3f}'.format(recall_train))
print('train set Specificity={:.3f}'.format(specificity_train))
print('train set F1={:.3f}'.format(f1_train))

In [13]:
df_train = pd.DataFrame({
    'ID':train['ID'],
    'True':y_ini,
    'Pre':pro_train
})
# Export predicted labels so downstream confusion-matrix analyses can reuse them.
df_train.to_csv('SVM_train.csv',index=False)

In [ ]:
# Import the libraries required for data handling, modeling, and evaluation.
import pandas as pd
from sklearn.metrics import (roc_auc_score, accuracy_score, 
                           precision_score, recall_score, 
                           # Report model performance metrics for this cohort.
                           f1_score, confusion_matrix)

test_files = [
    'test1.csv',
    'test2.csv',
    'test3.csv',
    'test4.csv',
    'test5.csv',
    'test6.csv',
    'test7.csv',
    'test8.csv'
]

for test_file in test_files:
    # Load the training and external validation datasets from CSV files.
    test = pd.read_csv(test_file)
    
    X_test = test.iloc[:, 1:6]
    y_test = test.iloc[:, 6]
    
    # Generate predicted probabilities for ROC-AUC evaluation.
    pro_test = model.predict_proba(X_test)[:, 1]
    # Generate class labels for accuracy and related classification metrics.
    y_pred_test = model.predict(X_test)
    
    auc_test = roc_auc_score(y_test, pro_test)
    accuracy_test = accuracy_score(y_test, y_pred_test)
    precision_test = precision_score(y_test, y_pred_test)
    recall_test = recall_score(y_test, y_pred_test)
    f1_test = f1_score(y_test, y_pred_test)
    
    confusion_test = confusion_matrix(y_test, y_pred_test)
    tn, fp, fn, tp = confusion_test.ravel()
    specificity_test = tn / (tn + fp)
    
    print(f'\nResults for {test_file}:')
    print(f'Test set AUC = {auc_test:.3f}')                   
    print(f'Test set Accuracy = {accuracy_test:.3f}')         
    print(f'Test set Precision = {precision_test:.3f}')       
    print(f'Test set Sensitivity (Recall) = {recall_test:.3f}')
    print(f'Test set Specificity = {specificity_test:.3f}')   
    print(f'Test set F1 = {f1_test:.3f}')                     
    
    df_test = pd.DataFrame({
        'ID': test['ID'],
        'True': y_test,
        'Pre': pro_test
    })
    # Export predicted labels so downstream confusion-matrix analyses can reuse them.
    df_test.to_csv(f'SVM_{test_file}_predictions.csv', index=False)

import joblib
model_filename = 'SVM_model.joblib'
# Save the trained model for later reuse.
joblib.dump(model, model_filename)